In [1]:
import os
import pandas as pd
import numpy as np

# weighting method 비교: FP-only vs IMG-only vs 고정 커스텀 가중치(FP=0.75/IMG=0.25,
# FP=0.25/IMG=0.75) vs UNIFORM(고정 0.5/0.5) vs CUE(softmax, 학습된 modality 가중치).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 소스: (표시 이름, metrics.csv 루트 경로). CUE는 항상 마지막에 오도록 순서를 고정한다.
model_sources = {
    'FP': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'FP'),
    'IMG': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'IMG'),
    'UNIFORM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'UNIFORM'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}

model_order = ['FP', 'IMG', 'UNIFORM', 'CUE']

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: 2 x 4 x 6 = 48).")

✅ 데이터 로드 완료: 32개의 항목이 수집되었습니다 (기대값: 2 x 4 x 6 = 48).


In [3]:
# metric별 rank 행렬: row = Dataset-Target(8개) + 마지막 'avg' 행,
# column = model_order(FP/IMG/CUSTOM_FP75_IMG25/UNIFORM/CUSTOM_FP25_IMG75/CUE).
def build_rank_matrix(df_all, metric, model_order=model_order):
    """지정한 metric에 대해 8개의 (Dataset, Target) 조합별로 model_order 각 모델의 순위를 담은
    행렬을 만든다. 마지막 'avg' 행은 각 모델의 8개 행별 순위 평균이다.

    Returns:
        pd.DataFrame: 컬럼이 'Dataset-Target', FP, IMG, CUSTOM_FP75_IMG25, UNIFORM,
            CUSTOM_FP25_IMG75, CUE 순인 표.
    """
    rows = {}
    for d_name in datasets:
        for target in targets:
            df_sub = df_all[(df_all['Dataset'] == d_name) & (df_all['Target'] == target)]
            df_sub = df_sub.set_index('Model').reindex(model_order)
            if metric not in df_sub.columns or df_sub[metric].isna().all():
                continue
            ranks = df_sub[metric].rank(ascending=LOWER_IS_BETTER.get(metric, True), method='min')
            rows[f'{d_name}-{target}'] = ranks

    rank_df = pd.DataFrame(rows).T
    rank_df = rank_df.reindex(columns=model_order)
    rank_df.loc['avg'] = rank_df.mean(axis=0).round(1)

    return rank_df.reset_index().rename(columns={'index': 'Dataset-Target'})


table_save_dir = os.path.join(cue_base, 'table', 'weight_comparison_v5')
os.makedirs(table_save_dir, exist_ok=True)

for metric in metrics_of_interest:
    rank_table = build_rank_matrix(df_total, metric)

    print(f"\n=== Rank table ({metric}) ===")
    print(rank_table.to_string(index=False))

    save_path = os.path.join(table_save_dir, f'{metric}.csv')
    rank_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"✅ 저장 완료: {save_path}")


=== Rank table (rmse) ===
Dataset-Target  FP  IMG  UNIFORM  CUE
         Kd-PI 3.0  4.0      2.0  1.0
       Kd-Ssel 1.0  4.0      3.0  2.0
Kd-WS(alpha=2) 3.0  4.0      2.0  1.0
    Kd-RS(k=2) 3.0  4.0      2.0  1.0
         Ki-PI 3.0  4.0      2.0  1.0
       Ki-Ssel 3.0  4.0      2.0  1.0
Ki-WS(alpha=2) 3.0  4.0      2.0  1.0
    Ki-RS(k=2) 1.0  4.0      3.0  2.0
           avg 2.5  4.0      2.2  1.2
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/weight_comparison_v5/rmse.csv

=== Rank table (r2) ===
Dataset-Target  FP  IMG  UNIFORM  CUE
         Kd-PI 3.0  4.0      2.0  1.0
       Kd-Ssel 1.0  4.0      3.0  2.0
Kd-WS(alpha=2) 3.0  4.0      2.0  1.0
    Kd-RS(k=2) 3.0  4.0      2.0  1.0
         Ki-PI 3.0  4.0      2.0  1.0
       Ki-Ssel 3.0  4.0      2.0  1.0
Ki-WS(alpha=2) 3.0  4.0      2.0  1.0
    Ki-RS(k=2) 1.0  4.0      3.0  2.0
           avg 2.5  4.0      2.2  1.2
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/weight_comparison_v5/r2.csv

=== R